In [1]:
import numpy as np
from skrebate import ReliefF
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, roc_curve, average_precision_score, 
                             cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# Load the dataset
data = pd.read_excel(r"C:\Users\PC\Desktop\PhD\Prospective Study\dataset\class 1\class1_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

for column in X.columns:
    X[column] = X[column].map(lambda x: 1 if x == True else (0 if x == False else x))

# Apply ReliefF
relief = ReliefF()
relief.fit(X.values, Y.values)

# Get the feature importances
feature_importances = relief.feature_importances_

# Sort features by importance
sorted_indices = np.argsort(feature_importances)[::-1]
sorted_features = [(index, feature_importances[index]) for index in sorted_indices]

classifiers = [
    ('Decision Tree', DecisionTreeClassifier()),
    ('Random Forest', RandomForestClassifier()),
    ('SVM', SVC(probability=True)),  # SVC by default doesn't have predict_proba, setting probability=True fixes that
    ('KNN', KNeighborsClassifier()),
    ('Naive Bayes', GaussianNB()),
    ('AdaBoost', AdaBoostClassifier()),
    ('Gradient Boosting', GradientBoostingClassifier()),
    ('MLP', MLPClassifier(max_iter=1000, random_state=42))  # Setting max_iter to 1000 to ensure convergence
]

# Initialize a dictionary to store the best AUC and feature combination for each algorithm
best_results_by_classifier = {
    'Decision Tree': (0, None, None),  # (AUC, feature_indices, fold)
    'Random Forest': (0, None, None),
    'SVM': (0, None, None),
    'KNN': (0, None, None),
    'Naive Bayes': (0, None, None),
    'AdaBoost': (0, None, None),
    'Gradient Boosting': (0, None, None),
    'MLP': (0, None, None)
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Iteratively select the top features based on ReliefF ranking
for num_features in range(1, 41):  # We'll use top 1 to top 10 features
    selected_features_indices = [index for index, _ in sorted_features[:num_features]]
    X_reduced = X.iloc[:, selected_features_indices]

    # Perform 10-fold cross-validation for each classification algorithm
    for name, model in classifiers:
        # Lists to store AUCs for each fold
        aucs = []

        # Loop through each fold
        for train_index, test_index in skf.split(X_reduced, Y):
            X_train_fold, X_test_fold = X_reduced.iloc[train_index], X_reduced.iloc[test_index]
            y_train_fold, y_test_fold = Y.iloc[train_index], Y.iloc[test_index]

            # Train the model on the current fold
            model.fit(X_train_fold, y_train_fold)

            # Get predicted probabilities for AUC calculation (if possible)
            if hasattr(model, "predict_proba"):
                y_pred_probs_fold = model.predict_proba(X_test_fold)[:, 1]
            else:
                y_pred_probs_fold = model.predict(X_test_fold)  # For models that don't have predict_proba

            # Calculate AUC for the current fold and store
            aucs.append(roc_auc_score(y_test_fold, y_pred_probs_fold))

        # Compare and store the best AUC for each classifier
        avg_auc = np.mean(aucs)
        if avg_auc > best_results_by_classifier[name][0]:
            best_results_by_classifier[name] = (avg_auc, selected_features_indices, np.argmax(aucs))

best_results_by_classifier

{'Decision Tree': (0.7295554278642444,
  [37, 30, 25, 26, 29, 28, 32, 38, 36, 10],
  9),
 'Random Forest': (0.7651690923067337,
  [37,
   30,
   25,
   26,
   29,
   28,
   32,
   38,
   36,
   10,
   7,
   13,
   33,
   12,
   34,
   24,
   14,
   18,
   3,
   15,
   6,
   16,
   8,
   11,
   17,
   4,
   1,
   9,
   19,
   31,
   5,
   0,
   20,
   35,
   22,
   23,
   2,
   27,
   21],
  7),
 'SVM': (0.7066250804328269,
  [37,
   30,
   25,
   26,
   29,
   28,
   32,
   38,
   36,
   10,
   7,
   13,
   33,
   12,
   34,
   24,
   14,
   18,
   3,
   15,
   6,
   16,
   8,
   11,
   17,
   4,
   1,
   9,
   19,
   31,
   5,
   0,
   20,
   35,
   22,
   23,
   2],
  9),
 'KNN': (0.6947941749136894,
  [37,
   30,
   25,
   26,
   29,
   28,
   32,
   38,
   36,
   10,
   7,
   13,
   33,
   12,
   34,
   24,
   14,
   18,
   3,
   15,
   6,
   16,
   8,
   11,
   17,
   4,
   1,
   9,
   19,
   31,
   5,
   0,
   20,
   35,
   22,
   23],
  6),
 'Naive Bayes': (0.6610515395860195,
 